# 🚗 Driver Behavior Recognition — Kaggle Complete Pipeline
**Repo:** [tunaleeZ/Driver_behaviour](https://github.com/tunaleeZ/Driver_behaviour)  
**Nhận diện 5 hành vi:** `normal`, `drowsy`, `sleepy`, `drinking`, `phone`  

---

### ⚙️ Hướng dẫn cài đặt trên Kaggle:
1. **Accelerator:** Vào menu bên phải (*Notebook options*) -> **Accelerator** -> Chọn **GPU T4 x2** (hoặc GPU P100).
2. **Internet:** Vào menu bên phải -> **Internet** -> Chọn **On** (cần thiết để clone repo từ GitHub và tải file trọng số MediaPipe Face Landmarker).
3. **Dataset:** Đã upload file zip chứa 2 video lên Kaggle và đặt tên dataset là **`driver_behaviour_dataset`**. Khi đó dataset sẽ tự động được mount vào `/kaggle/input/driver-behaviour-dataset`.


In [ ]:
# ==========================================
# 1. KIỂM TRA MÔI TRƯỜNG & GPU
# ==========================================
import sys, os, subprocess
from pathlib import Path
import torch

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Model:", torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM khả dụng: {vram:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào Notebook options -> Accelerator -> Chọn GPU T4 x2.")


## 2. Clone Repository & Cài Đặt Dependencies
Clone toàn bộ mã nguồn repo GitHub vào `/kaggle/working` và cài đặt gói dự án `dbm` cùng các thư viện bổ trợ.


In [ ]:
# ==========================================
# 2. CLONE GITHUB REPO & CÀI ĐẶT
# ==========================================
REPO_URL = "https://github.com/tunaleeZ/Driver_behaviour.git"
WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "Driver_behaviour"

if not REPO_DIR.exists():
    print(f"Đang clone repo từ {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repo đã có sẵn tại {REPO_DIR}")

# Chuyển thư mục làm việc về REPO_DIR
os.chdir(REPO_DIR)
print("Thư mục làm việc hiện tại:", os.getcwd())

# Cài đặt các gói phụ thuộc bổ sung (Kaggle đã có sẵn torch, numpy, opencv)
print("Đang cài đặt thư viện cần thiết...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe>=0.10.0", "imageio-ffmpeg>=0.4.9", "openpyxl>=3.1.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"], check=True)

# Thêm src vào sys.path
sys.path.insert(0, str(REPO_DIR / "src"))

import mediapipe as mp
import dbm
print("MediaPipe version:", mp.__version__)
print("Package dbm đã import thành công!")


## 3. Tải Pretrained Model MediaPipe Face Landmarker
Model trọng số `face_landmarker.task` thuộc diện `.gitignore` trên GitHub. Đoạn mã này sẽ tự động tải trực tiếp từ Google Storage chính thức của MediaPipe.


In [ ]:
# ==========================================
# 3. TẢI MODEL FACE LANDMARKER (.task)
# ==========================================
import urllib.request

MODELS_DIR = REPO_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
LANDMARKER_PATH = MODELS_DIR / "face_landmarker.task"
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"

if not LANDMARKER_PATH.exists() or LANDMARKER_PATH.stat().st_size == 0:
    print(f"Đang tải face_landmarker.task từ Google Storage...")
    urllib.request.urlretrieve(MODEL_URL, LANDMARKER_PATH)
    file_size_mb = LANDMARKER_PATH.stat().st_size / 1e6
    print(f"Tải thành công: {LANDMARKER_PATH.name} ({file_size_mb:.2f} MB)")
else:
    file_size_mb = LANDMARKER_PATH.stat().st_size / 1e6
    print(f"Model đã có sẵn: {LANDMARKER_PATH.name} ({file_size_mb:.2f} MB)")

# Đặt biến môi trường để toàn bộ code trích xuất tự động dùng model này
os.environ["DBM_FACE_LANDMARKER_PATH"] = str(LANDMARKER_PATH)


## 4. Quét & Đồng Bộ 2 Video Đã Upload Vào `raw/public/`
Tự động tìm kiếm 2 video trong `/kaggle/input` (hỗ trợ cả trường hợp Kaggle tự giải nén hoặc còn giữ file zip) và đưa vào đúng vị trí `driver_behavior_dataset_v01/raw/public/`.


In [ ]:
# ==========================================
# 4. TÌM VIDEO TỪ KAGGLE DATASET VÀ COPY VÀO RAW
# ==========================================
import shutil, zipfile

KAGGLE_INPUT = Path("/kaggle/input")
RAW_PUBLIC_DIR = REPO_DIR / "driver_behavior_dataset_v01" / "raw" / "public"
RAW_PUBLIC_DIR.mkdir(parents=True, exist_ok=True)

video_extensions = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
found_videos = []

# Tìm kiếm toàn bộ video hoặc file zip trong /kaggle/input
for root, _, files in os.walk(KAGGLE_INPUT):
    for f in files:
        p = Path(root) / f
        if p.suffix.lower() in video_extensions:
            found_videos.append(p)
        elif p.suffix.lower() == ".zip":
            print(f"Phát hiện file zip: {p.name}, đang giải nén...")
            with zipfile.ZipFile(p, "r") as z:
                for member in z.namelist():
                    if Path(member).suffix.lower() in video_extensions:
                        print(f"  -> Giải nén: {member}")
                        z.extract(member, RAW_PUBLIC_DIR)
                        extracted = RAW_PUBLIC_DIR / member
                        if extracted.parent != RAW_PUBLIC_DIR:
                            shutil.move(str(extracted), str(RAW_PUBLIC_DIR / extracted.name))

# Cập nhật danh sách video
found_videos = []
for root, _, files in os.walk(KAGGLE_INPUT):
    for f in files:
        p = Path(root) / f
        if p.suffix.lower() in video_extensions:
            found_videos.append(p)

print(f"\nTìm thấy {len(found_videos)} file video trong /kaggle/input:")
for v in found_videos:
    size_mb = v.stat().st_size / 1e6
    print(f"  - {v.name} ({size_mb:.2f} MB)")
    dst = RAW_PUBLIC_DIR / v.name
    if not dst.exists():
        try:
            os.symlink(v, dst)
        except OSError:
            shutil.copy2(v, dst)

available_videos = list(RAW_PUBLIC_DIR.glob("*.mp4")) + list(RAW_PUBLIC_DIR.glob("*.avi"))
print(f"\nDanh sách video sẵn sàng trong {RAW_PUBLIC_DIR.name}:")
for v in available_videos:
    print(f"  [OK] {v.name}")


## 5. Trích Xuất Vài Chục Giây Cuối Video Để Làm Test Video Độc Lập
Do dữ liệu ít, ta trích xuất **60 giây cuối cùng** của 1 trong 2 video làm video kiểm thử độc lập (`test_unseen_clip.mp4`).
Phần này sẽ hoàn toàn bị loại khỏi tập huấn luyện/validation để đảm bảo tuyệt đối không có rò rỉ dữ liệu (No Data Leakage).
Sau khi huấn luyện xong mô hình, ta sẽ chạy trực tiếp mô hình trên video này và xuất ra video có gắn nhãn thời gian thực để bạn tải về.


In [ ]:
# ==========================================
# 5. TRÍCH XUẤT ĐOẠN CUỐI LÀM TEST VIDEO
# ==========================================
import cv2
from pathlib import Path

TEST_DURATION_SEC = 60  # Số giây cuối muốn trích làm bài test độc lập
TEST_VIDEO_PATH = WORK_DIR / "test_unseen_clip.mp4"

# Đảm bảo lấy danh sách video trực tiếp từ thư mục raw/public
RAW_PUBLIC_DIR = REPO_DIR / "driver_behavior_dataset_v01" / "raw" / "public"
available_videos = sorted(list(RAW_PUBLIC_DIR.glob("*.mp4")) + list(RAW_PUBLIC_DIR.glob("*.avi")))

if not available_videos:
    raise RuntimeError(f"Chưa có video nào trong {RAW_PUBLIC_DIR}. Hãy chạy lại Mục 4 trước!")

# Chọn video đầu tiên để trích xuất đoạn cuối
target_raw_video = available_videos[0]
cap = cv2.VideoCapture(str(target_raw_video))
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_dur = total_frames / fps
cap.release()

test_start_sec = max(0.0, video_dur - TEST_DURATION_SEC)
print(f"Video được chọn: {target_raw_video.name}")
print(f"Tổng thời lượng: {video_dur:.2f}s ({video_dur/60:.2f} phút) @ {fps:.1f} FPS")
print(f"Đoạn trích làm Test: từ {test_start_sec:.2f}s đến {video_dur:.2f}s ({TEST_DURATION_SEC}s)")

# Cắt đoạn test bằng ffmpeg cực nhanh (không re-encode)
cmd_cut_test = [
    "ffmpeg", "-y",
    "-ss", str(test_start_sec),
    "-i", str(target_raw_video),
    "-t", str(TEST_DURATION_SEC),
    "-c", "copy",
    str(TEST_VIDEO_PATH)
]
subprocess.run(cmd_cut_test, check=True)
print(f"✓ Đã tạo thành công video kiểm thử độc lập: {TEST_VIDEO_PATH.name} ({TEST_VIDEO_PATH.stat().st_size / 1e6:.2f} MB)")


## 6. Đồng Bộ & Lọc Metadata
Tự động lọc `metadata.xlsx`:
1. Giữ lại các dòng thuộc về 2 video hiện có.
2. Loại bỏ bất kỳ segment nào nằm trong khoảng thời gian của đoạn test vừa trích (đảm bảo chống data leakage 100%).


In [ ]:
# ==========================================
# 6. LỌC METADATA CHO TRAIN/VAL & TRÍCH XUẤT GROUNDTRUTH CHO TEST
# ==========================================
import pandas as pd
from pathlib import Path

RAW_PUBLIC_DIR = REPO_DIR / "driver_behavior_dataset_v01" / "raw" / "public"
META_XLSX = REPO_DIR / "driver_behavior_dataset_v01" / "annotations" / "metadata.xlsx"
META_CSV = REPO_DIR / "driver_behavior_dataset_v01" / "annotations" / "metadata.csv"
TEST_GT_CSV = REPO_DIR / "outputs" / "test_groundtruth.csv"
TEST_GT_CSV.parent.mkdir(parents=True, exist_ok=True)

def time_to_sec(val):
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip()
    parts = val_str.split(":")
    if len(parts) == 3:
        return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
    elif len(parts) == 2:
        return float(parts[0]) * 60 + float(parts[1])
    return float(parts[0])

# Đọc metadata gốc
df_meta = pd.read_excel(META_XLSX) if META_XLSX.exists() else pd.read_csv(META_CSV)

# segment_id PHẢI duy nhất (cut_segments lưu segments/<label>/<segment_id>.mp4).
# Nếu xlsx đang dùng mã lớp (SEG000001=normal, SEG000002=phone, ...) thì sinh ID duy nhất theo thứ tự dòng.
df_meta = df_meta.dropna(how="all").reset_index(drop=True)
if df_meta["segment_id"].isna().any() or df_meta["segment_id"].duplicated().any():
    print("⚠️ segment_id bị trùng/trống (đang dùng mã lớp) -> sinh lại ID duy nhất SEG000001.. theo thứ tự dòng")
    df_meta["segment_id"] = [f"SEG{i:06d}" for i in range(1, len(df_meta) + 1)]
print(f"Metadata gốc: {len(df_meta)} rows ({df_meta['source_video'].nunique()} videos)")

raw_root = REPO_DIR / "driver_behavior_dataset_v01" / "raw"
# 1. Chỉ giữ lại các video thực tế có trong thư mục raw/
existing_mask = df_meta["source_video"].apply(lambda v: (raw_root / str(v).strip()).exists())
df_available = df_meta[existing_mask].copy()

# 2. TRÍCH XUẤT GROUNDTRUTH CHO ĐOẠN 60s TEST
df_available["start_sec_calc"] = df_available["start_time"].apply(time_to_sec)
df_available["end_sec_calc"] = df_available["end_time"].apply(time_to_sec)
target_rel_name = f"public/{target_raw_video.name}"

# Các segment giao với khoảng thời gian test [test_start_sec, video_dur]
test_mask = (df_available["source_video"] == target_rel_name) & (df_available["end_sec_calc"] > test_start_sec)
df_test_gt = df_available[test_mask].copy()

# Quy đổi mốc thời gian về [0.0s -> 60.0s] tương ứng với video test
df_test_gt["test_clip_start"] = (df_test_gt["start_sec_calc"] - test_start_sec).clip(lower=0.0)
df_test_gt["test_clip_end"] = (df_test_gt["end_sec_calc"] - test_start_sec).clip(upper=TEST_DURATION_SEC)
df_test_gt["test_clip_duration"] = df_test_gt["test_clip_end"] - df_test_gt["test_clip_start"]

cols_to_save = ["segment_id", "label", "test_clip_start", "test_clip_end", "test_clip_duration", "original_start", "original_end"]
df_test_gt["original_start"] = df_test_gt["start_time"]
df_test_gt["original_end"] = df_test_gt["end_time"]
df_test_gt[cols_to_save].to_csv(TEST_GT_CSV, index=False)

print(f"\n✓ ĐÃ TRÍCH XUẤT GROUND TRUTH CHO TEST VIDEO ({len(df_test_gt)} segments) -> {TEST_GT_CSV.name}:")
for _, r in df_test_gt.iterrows():
    print(f"  [{r['test_clip_start']:.1f}s -> {r['test_clip_end']:.1f}s]: nhãn '{r['label']}' (gốc: {r['original_start']} -> {r['original_end']})")

# 3. Tách hoàn toàn đoạn test ra khỏi tập Train/Val (chống Data Leakage)
df_available = df_available[~test_mask].drop(columns=["start_sec_calc", "end_sec_calc"])

print(f"\nSố dòng metadata còn lại dùng cho Train/Val: {len(df_available)} rows")
print(f"Source videos khả dụng: {df_available['source_video'].unique().tolist()}")
print(f"Subjects: {df_available['subject_id'].unique().tolist()}")
print("\nPhân bố nhãn hành vi trong tập Train/Val:")
print(df_available["label"].value_counts())

# Ghi cả Excel (đặt đúng sheet_name='metadata') và CSV
df_available.to_excel(META_XLSX, sheet_name="metadata", index=False)
df_available.to_csv(META_CSV, index=False)
print(f"\n✓ Đã cập nhật xong metadata cho Train/Val: {META_XLSX.name} và {META_CSV.name}")


## 7. Kiểm Kê & Xác Thực Metadata (Pipeline Step 1)
Chạy `inventory.py` để quét thông số video (fps, duration, resolution) và `validate_metadata.py` để kiểm tra toàn vẹn dữ liệu.


In [ ]:
# ==========================================
# 7. INVENTORY & VALIDATE METADATA
# ==========================================
print("=== CHẠY INVENTORY.PY ===")
subprocess.run([
    sys.executable, "driver_behavior_dataset_v01/scripts/inventory.py",
    "--overwrite",
    "--verbose"
], check=True)

print("\n=== CHẠY VALIDATE_METADATA.PY ===")
subprocess.run([
    sys.executable, "driver_behavior_dataset_v01/scripts/validate_metadata.py",
    "--input", "driver_behavior_dataset_v01/annotations/metadata.csv",
    "--verbose"
], check=True)


## 8. Cắt Video Thành Các Segment Hành Vi (Pipeline Step 2)
Dùng FFmpeg cắt các đoạn hành vi theo start/end timestamp trong metadata và lưu vào `segments/<label>/`.


In [ ]:
# ==========================================
# 8. CẮT VIDEO SEGMENTS (FFMPEG)
# ==========================================
print("=== CẮT VIDEO SEGMENTS ===")
subprocess.run([
    sys.executable, "driver_behavior_dataset_v01/scripts/cut_segments.py",
    "--workers", "2",
    "--overwrite",
    "--verbose"
], check=True)

segments_dir = REPO_DIR / "driver_behavior_dataset_v01" / "segments"
print("\nThống kê số lượng video clip segment đã cắt:")
for folder in sorted(segments_dir.iterdir()):
    if folder.is_dir():
        clips = list(folder.glob("*.mp4"))
        print(f"  - Lớp {folder.name:12s}: {len(clips):3d} video clips")


## 9. Phân Chia Tập Train / Val (Pipeline Step 3)
Phân chia độc lập theo Subject (hoặc theo Video nếu 2 video cùng 1 subject) để đảm bảo:
- Cả `train` và `val` đều có dữ liệu cửa sổ cho quá trình học và kiểm định dừng sớm (early stopping).
- Không có rò rỉ giữa tập học và tập kiểm tra.


In [ ]:
# ==========================================
# 9. TẠO SPLITS TRAIN / VAL
# ==========================================
df_curr_meta = pd.read_csv(META_CSV)
valid_meta = df_curr_meta[df_curr_meta["quality"].str.lower() != "bad"]
unique_subs = valid_meta["subject_id"].unique()

SPLITS_CSV = REPO_DIR / "driver_behavior_dataset_v01" / "annotations" / "splits.csv"

if len(unique_subs) >= 2:
    print(f"Có {len(unique_subs)} subject ({unique_subs.tolist()}). Chia tập theo subject:")
    subprocess.run([
        sys.executable, "driver_behavior_dataset_v01/scripts/make_splits.py",
        "--train-ratio", "0.6",
        "--val-ratio", "0.4",
        "--test-ratio", "0.0",
        "--overwrite"
    ], check=True)
else:
    print(f"Chỉ có 1 subject ({unique_subs[0]}). Phân bổ theo từng video vào Train và Val:")
    videos = valid_meta["source_video"].unique()
    train_vid = videos[0]
    val_vid = videos[1] if len(videos) > 1 else videos[0]
    
    valid_meta["original_subject_id"] = valid_meta["subject_id"]
    valid_meta["subject_id"] = valid_meta["source_video"].apply(
        lambda v: f"{unique_subs[0]}_A" if v == train_vid else f"{unique_subs[0]}_B"
    )
    valid_meta.to_csv(META_CSV, index=False)
    
    splits_df = pd.DataFrame([
        {"subject_id": f"{unique_subs[0]}_A", "split": "train"},
        {"subject_id": f"{unique_subs[0]}_B", "split": "val"}
    ])
    splits_df.to_csv(SPLITS_CSV, index=False)

splits_df = pd.read_csv(SPLITS_CSV)
print("\nKết quả phân chia (splits.csv):")
print(splits_df)


## 10. Trích Xuất MediaPipe Face Landmarks (Pipeline Step 4)
Chạy MediaPipe Face Landmarker trích xuất 478 tọa độ khuôn mặt 3D theo thời gian thực và lưu thành các file `.npz` đặc trưng hình học.


In [ ]:
# ==========================================
# 10. TRÍCH XUẤT MEDIA PIPE LANDMARKS (.npz)
# ==========================================
print("=== TRÍCH XUẤT ĐẶC TRƯNG LANDMARKS ===")
subprocess.run([
    sys.executable, "driver_behavior_dataset_v01/scripts/extract_landmarks.py",
    "--model-path", str(LANDMARKER_PATH),
    "--verbose"
], check=True)

landmarks_dir = REPO_DIR / "driver_behavior_dataset_v01" / "features" / "landmarks"
npz_files = list(landmarks_dir.glob("*.npz"))
print(f"\nĐã trích xuất thành công {len(npz_files)} file .npz vào {landmarks_dir}")

import numpy as np
if npz_files:
    sample = np.load(npz_files[0])
    print(f"\nKiểm tra cấu trúc file mẫu ({npz_files[0].name}):")
    for k in sample.files:
        print(f"  - {k:20s}: shape={sample[k].shape}, dtype={sample[k].dtype}")


## 11. Huấn Luyện Mô Hình (Landmark-LSTM)
Huấn luyện mô hình chuỗi thời gian LSTM trên đặc trưng landmark với trọng số cân bằng lớp và early stopping.


In [ ]:
# ==========================================
# 11. HUẤN LUYỆN MODEL LANDMARK-LSTM
# ==========================================
CONFIG_PATH = "configs/models/landmark_lstm.yaml"
RUN_NAME = "landmark_lstm_run"

print(f"Bắt đầu huấn luyện mô hình với cấu hình: {CONFIG_PATH}...")
subprocess.run([
    sys.executable, "-m", "dbm.train",
    "--config", CONFIG_PATH,
    "--run-name", RUN_NAME,
    "--override", "batch_size=16",
    "--override", "max_epochs=25",
    "--override", "lr=0.001"
], check=True)


## 12. Đánh Giá & Vẽ Biểu Đồ (Loss Curves & Confusion Matrix)


In [ ]:
# ==========================================
# 12. ĐÁNH GIÁ VÀ HIỂN THỊ ĐỒ THỊ
# ==========================================
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = REPO_DIR / "outputs" / RUN_NAME
best_ckpt = run_dir / "best.pt"
print(f"Đánh giá checkpoint tốt nhất: {best_ckpt}")

# Đánh giá lại trên tập val
subprocess.run([
    sys.executable, "-m", "dbm.evaluate",
    "--checkpoint", str(best_ckpt),
    "--split", "val"
], check=True)

# Hiển thị biểu đồ huấn luyện và ma trận nhầm lẫn
fig, axs = plt.subplots(1, 2, figsize=(16, 6))

lc_img_path = run_dir / "learning_curves.png"
if lc_img_path.exists():
    img1 = mpimg.imread(str(lc_img_path))
    axs[0].imshow(img1)
    axs[0].set_title("Biểu Đồ Học Tập (Train/Val Loss & Macro-F1)", fontsize=14)
    axs[0].axis("off")

cm_img_path = run_dir / "confusion_matrix.png"
if cm_img_path.exists():
    img2 = mpimg.imread(str(cm_img_path))
    axs[1].imshow(img2)
    axs[1].set_title("Ma Trận Nhầm Lẫn (Validation Confusion Matrix)", fontsize=14)
    axs[1].axis("off")

plt.tight_layout()
plt.show()


## 13. Test Trực Tiếp Trên Video Test Độc Lập & Render Video Kết Quả
Cell này chạy suy luận trượt (Sliding-window Inference) trực tiếp trên file video test `test_unseen_clip.mp4` đã tách ở bước 5:
1. Xuất file bảng thống kê thời gian thực `outputs/test_timeline_predictions.csv` (thời điểm, nhãn dự đoán, xác suất từng lớp).
2. Vẽ nhãn dự đoán và thanh xác suất trực tiếp lên video và lưu file `outputs/test_annotated_video.mp4` để tải về xem trực quan!


In [ ]:
# ==========================================
# 13. TEST SUY LUẬN & SO SÁNH VỚI GROUND TRUTH
# ==========================================
csv_output = run_dir / "test_timeline_predictions.csv"
annotated_video_out = run_dir / "test_annotated_video.mp4"
test_gt_csv = REPO_DIR / "outputs" / "test_groundtruth.csv"

print(f"Đang suy luận trượt trên video: {TEST_VIDEO_PATH.name}...")
subprocess.run([
    sys.executable, "-m", "dbm.infer_video",
    "--video", str(TEST_VIDEO_PATH),
    "--checkpoint", str(best_ckpt),
    "--output", str(csv_output),
    "--smoothing", "ema"
], check=True)

df_preds = pd.read_csv(csv_output)
df_gt = pd.read_csv(test_gt_csv) if test_gt_csv.exists() else pd.DataFrame()

# Hàm lấy nhãn Ground Truth tại thời điểm t
def get_gt_label(t):
    if df_gt.empty:
        return "unknown"
    match = df_gt[(df_gt["test_clip_start"] <= t) & (df_gt["test_clip_end"] >= t)]
    return match.iloc[-1]["label"] if not match.empty else "normal"

# Thêm cột Ground Truth vào bảng dự đoán để so sánh
df_preds["mid_sec"] = (df_preds["start_seconds"] + df_preds["end_seconds"]) / 2.0
df_preds["ground_truth"] = df_preds["mid_sec"].apply(get_gt_label)
df_preds["is_correct"] = df_preds["prediction"] == df_preds["ground_truth"]
df_preds.to_csv(csv_output, index=False)

accuracy = df_preds["is_correct"].mean() * 100
print(f"\n✓ Đã đối chiếu với Ground Truth: Độ chính xác trên đoạn test 60s = {accuracy:.2f}%")
print("\nBảng so sánh Ground Truth vs Prediction (10 cửa sổ đầu tiên):")
print(df_preds[["start_seconds", "end_seconds", "ground_truth", "prediction", "is_correct"]].head(10))

# --- RENDER VIDEO HIỂN THỊ CẢ GROUND TRUTH VÀ PREDICTION ---
print(f"\nĐang vẽ Ground Truth & Prediction lên video -> {annotated_video_out.name}...")
cap = cv2.VideoCapture(str(TEST_VIDEO_PATH))
v_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(str(annotated_video_out), cv2.VideoWriter_fourcc(*"mp4v"), v_fps, (width, height))

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    current_sec = frame_idx / v_fps
    
    # Lấy prediction và GT tại frame hiện tại
    active_pred = df_preds[(df_preds["start_seconds"] <= current_sec) & (df_preds["end_seconds"] >= current_sec)]
    pred_label = active_pred.iloc[-1]["prediction"] if not active_pred.empty else "analyzing..."
    gt_label = get_gt_label(current_sec)
    
    is_match = (pred_label == gt_label)
    status_text = "CORRECT" if is_match else "MISMATCH"
    status_color = (0, 255, 0) if is_match else (0, 0, 255)  # Xanh nếu đúng, Đỏ nếu sai
    
    # Vẽ HUD hiển thị thông tin
    cv2.rectangle(frame, (20, 20), (520, 130), (0, 0, 0), -1)
    cv2.putText(frame, f"Time: {current_sec:.1f}s / {TEST_DURATION_SEC}s", (35, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    cv2.putText(frame, f"GT   (Thuc te): {gt_label.upper()}", (35, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
    cv2.putText(frame, f"PRED (Du doan): {pred_label.upper()} [{status_text}]", (35, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, status_color, 2)
    
    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()
print(f"✓ Đã tạo video so sánh trực quan thành công: {annotated_video_out.name} ({annotated_video_out.stat().st_size / 1e6:.2f} MB)")


## 14. Nén Toàn Bộ Kết Quả & Video Test Để Tải Về


In [ ]:
# ==========================================
# 14. NÉN KẾT QUẢ, MODEL & VIDEO TEST ĐỂ TẢI VỀ
# ==========================================
zip_out = WORK_DIR / "driver_behaviour_results.zip"
print(f"Đang nén toàn bộ outputs và video test vào {zip_out}...")
subprocess.run(["zip", "-r", str(zip_out), "outputs/"], check=True)
print(f" Hoàn tất! Bạn có thể tải file '{zip_out.name}' trực tiếp tại panel Output bên phải của Kaggle.")
